# Reinforcement learning with verifiable rewards

Supervised fine-tuning is easy to explain: show the model the right answer and make that answer more likely. But for a lot of the problems we care about, nobody has written the right answer down. What we do have is a way to check one. Code either passes its tests or it doesn't, and a maths answer either matches or it doesn't.

A check is enough to learn from. Let the model try a few times, check each try, and nudge it towards the ones that passed. That's reinforcement learning with verifiable rewards, RLVR for short, and it's the recipe behind DeepSeek-R1's reasoning ([DeepSeek-AI, 2025](https://arxiv.org/abs/2501.12948)).

![The RLVR loop. The policy writes four answers to the prompt 7 2 9 4, a checker scores each one, and the update makes the answers that scored above the group's mean more likely.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/rlvr-loop.svg)

That picture is the whole method. We'll build it with GRPO, the variant DeepSeekMath introduced ([Shao et al., 2024](https://arxiv.org/abs/2402.03300)) and R1 was trained with, first on problems small enough to check by hand and then on a tiny model trained with Dew's `GRPOObjective`. It all runs on a CPU in about a minute.

## Setup

We install Dew from GitHub; nothing in this notebook needs a GPU. The sizes in the settings cell suit a toy this small, and its comments say where DeepSeekMath's recipe used something else.

In [ ]:
%pip install -q "dewml @ git+https://github.com/AshishKumar4/dew"

In [1]:
import os

VOCAB = 10           # the tokens are the digits 0 to 9
PROMPT_LENGTH = 4    # digits the policy reads
NEW_TOKENS = 6       # digits it writes
PROMPTS = 8          # prompts per batch
GROUPS = 8           # answers per prompt, G (DeepSeekMath used 64)
STEPS = 80           # rollouts in the training run
UPDATES = 2          # gradient steps on each rollout (DeepSeekMath took 1)
LEARNING_RATE = 0.05
BETA = 0.02          # weight of the KL penalty
NUDGES = 24          # updates in the one-choice example
SEED = 0

# The tutorial test sets DEW_TUTORIAL_SMOKE=1 to run every cell in minutes on a CPU.
if os.environ.get("DEW_TUTORIAL_SMOKE") == "1":
    PROMPTS, GROUPS, STEPS = 4, 4, 5

In [2]:
from types import SimpleNamespace

import dew  # before JAX's backend opens, so its numerical settings apply
import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import linen as nn

from dew import figures
from dew.data.prompts import INFO_KEY, LENGTH_KEY, PROMPT_KEY, SOURCE_KEY, TRUTH_KEY
from dew.nn.protocols import OutputTable
from dew.objectives.base import Step
from dew.objectives.rl import GRPOObjective, SampledRollout
from dew.rl import clipped_surrogate_terms, k3_kl
from dew.sampling import Sampling, generate

np.set_printoptions(precision=3, suppress=True)

/mnt/local/dew/envs/dew-cpu-32bbc37eb3a0/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## A task we can check

We want a task that's easy to check and that the model can't already do. The model reads four random digits and writes six more, and the six should keep counting up from the last digit it read, wrapping 9 round to 0. After `7 2 9 4`, the right answer is `5 6 7 8 9 0`.

The model never sees that rule. All it gets back is a score, the fraction of its six digits that are one more than the digit before. Below is that checker, scoring three answers to `7 2 9 4`.

In [3]:
def right_digits(previous, digits):
    """Which written digits are one more than the digit before them."""
    marks = []
    for digit in digits:
        marks.append(digit == (previous + 1) % VOCAB)
        previous = digit
    return marks


# Every Dew reward function takes four strings: where the prompt came from, the
# completion, the ground truth (here the prompt's last digit) and extra information.
def counts_up(source, completion, truth, info):
    digits = [int(digit) for digit in completion.split()]
    return float(np.mean(right_digits(int(truth), digits)))


def colors(marks):
    return ["accent_fill" if right else "warm_fill" for right in marks]


answers = ["5 6 7 8 9 0", "5 6 1 2 9 0", "3 1 4 1 5 9"]
marks = [right_digits(4, map(int, answer.split())) for answer in answers]
figures.tokens([["7", "2", "9", "4", *answer.split()] for answer in answers],
               fills=[["none"] * 4 + colors(right) for right in marks],
               notes=[f"reward {counts_up('', a, '4', ''):.2f}" for a in answers])

Figure(404.6 x 117, 3 shapes)

The first answer is perfect. The second slips twice, writing 1 where 7 belongs and 9 where 3 belongs, so it scores 4 out of 6, or 0.67. That partial credit gives a model that starts out writing nonsense a way to tell a slightly better answer from a worse one.

## The model we'll train

Our model is about as simple as a language model gets. It's a 10 by 10 table of logits, and the row for the digit it just read scores every possible next digit. A bigram model like this can learn the counting rule perfectly, since the right next digit depends only on the last one.

`SampledRollout` asks the model for `GROUPS` answers to each prompt, runs the checker on all of them, and packs a batch `GRPOObjective` can train on. That batch's 64 rows of 10 tokens are 8 prompts times 8 answers.

In [4]:
class Bigram(nn.Module):
    """Logits for the next digit: one row of a table per digit just read."""

    vocab_size: int

    def setup(self):
        shape = (self.vocab_size, self.vocab_size)
        self.table = self.param("table", nn.initializers.normal(0.1), shape)

    # Dew's losses and samplers read a decoder through two methods: the states its head
    # reads, here the one-hot digits, and the head itself, here the table.
    def hidden_states(self, tokens, train=False, **fields):
        return jax.nn.one_hot(tokens, self.vocab_size)

    def output_table(self):
        return OutputTable(self.table, vocab_major=False)

    # Generation threads a cache through every model; a bigram keeps nothing in it.
    @nn.compact
    def init_cache(self, batch_size):
        self.variable("cache", "index", lambda: jnp.zeros((batch_size,), jnp.int32))

    def __call__(self, tokens, train=False, decode=False, attention_mask=None, **rest):
        return self.hidden_states(tokens) @ self.table


# A prompt batch is all arrays, so the checker's strings travel as rows of UTF-8 bytes.
def text_rows(texts, width=8):
    rows = np.zeros((len(texts), width), np.int32)
    for row, text in enumerate(texts):
        encoded = np.frombuffer(text.encode(), np.uint8)
        rows[row, :len(encoded)] = encoded
    return rows


def prompt_batch(rng):
    prompts = rng.integers(0, VOCAB, (PROMPTS, PROMPT_LENGTH)).astype(np.int32)
    return {PROMPT_KEY: prompts,
            LENGTH_KEY: np.full(PROMPTS, PROMPT_LENGTH, np.int32),
            SOURCE_KEY: text_rows(["count"] * PROMPTS),
            TRUTH_KEY: text_rows([str(prompt[-1]) for prompt in prompts]),
            INFO_KEY: text_rows([""] * PROMPTS)}


policy = Bigram(VOCAB)
# beta > 0 keeps a frozen copy of the starting weights, for the KL penalty further down.
objective = GRPOObjective(policy, seq_len=PROMPT_LENGTH + NEW_TOKENS - 1, beta=BETA)
start = objective.init(jax.random.key(SEED))
# Samples GROUPS answers per prompt, scores them with counts_up, computes the advantages
# and packs the batch GRPOObjective reads.
rollout = SampledRollout(objective, counts_up, groups=GROUPS, max_new_tokens=NEW_TOKENS,
                         sampling=Sampling(temperature=1.0))
rng = np.random.default_rng(SEED)
untrained = SimpleNamespace(variables=start, updates=0)
batch = rollout(untrained, prompt_batch(rng), jax.random.key(1))
print({name: value.shape for name, value in batch.items()})

{'input_ids': (64, 10), 'text_segment_ids': (64, 10), 'text_positions': (64, 10), 'response_mask': (64, 10), 'behavior_log_probs': (64, 10), 'versions': (64, 10), 'session_index': (64, 10), 'call_index': (64, 10), 'advantages': (64, 10), 'session_weights': (64, 10), 'old_log_probs': (64, 10)}


## Start with a single choice

Before we touch whole sequences, let's shrink the problem as far as it goes. The model has to pick one of five answers, and only answer 3 is right. Its policy is a softmax over five numbers, $\pi_\theta = \mathrm{softmax}(\theta)$. How should θ move?

Our first try is to sample 8 answers and nudge θ towards each one that was right. The cell below runs 24 rounds of that and animates the probabilities.

In [5]:
K, RIGHT = 5, 3
reward = (jnp.arange(K) == RIGHT).astype(jnp.float32)
start_theta = jax.random.normal(jax.random.key(SEED), (K,)) * 0.5


def reinforce(theta, key, samples=8):
    answers = jax.random.categorical(key, theta, shape=(samples,))
    push = jax.nn.one_hot(answers, K) - jax.nn.softmax(theta)  # grad of log pi(answer)
    return jnp.mean(reward[answers][:, None] * push, axis=0)


theta, history = start_theta, [jax.nn.softmax(start_theta)]
for nudge in range(NUDGES):
    theta = theta + 1.0 * reinforce(theta, jax.random.key(100 + nudge))
    history.append(jax.nn.softmax(theta))

shown = range(0, NUDGES + 1, NUDGES // 8)
figures.bars([history[i] for i in shown], labels=[f"a={k}" for k in range(K)],
             captions=[f"update {i}" for i in shown], y_label="π(a)",
             colors=["tint1"] * RIGHT + ["accent"] + ["tint1"] * (K - RIGHT - 1))

Figure(460 x 260, 66 shapes)

Answer 3 starts at a probability of 0.12, behind three of the wrong answers, and 24 updates later it's at 0.94. Why does nudging towards the right answers work?

We'd like to raise the expected reward $J(\theta) = \sum_a \pi_\theta(a)\, R(a)$, but a checker has no gradient; you can't differentiate "the tests passed". There's a neat way around that. Because $\nabla \pi = \pi\, \nabla \log \pi$, the gradient of the expected reward becomes an average over the model's own tries:

$$
\nabla J = \mathbb{E}_{a \sim \pi_\theta}\big[R(a)\, \nabla \log \pi_\theta(a)\big].
$$

Read it out loud: sample an answer, look at its reward, and push its log-probability up by that much. That's REINFORCE ([Williams, 1992](https://link.springer.com/article/10.1007/BF00992696)), and the rest of this notebook is refinements of this one line. For a softmax the push is $\nabla_\theta \log \pi_\theta(a) = \mathrm{onehot}(a) - \pi_\theta$, the `push` line in the code above, which raises the sampled answer and lowers every answer in proportion to how likely it already was.

This problem is small enough for `jax.grad` to give us the exact gradient, so we can see how good an estimate from 8 tries is.

In [6]:
def expected_reward(theta):
    return jnp.sum(jax.nn.softmax(theta) * reward)


print("exact gradient  ", jax.grad(expected_reward)(start_theta))
print("100,000 samples ", reinforce(start_theta, jax.random.key(1), 100_000))
print("8 samples       ", reinforce(start_theta, jax.random.key(2), 8))

exact gradient   [-0.035 -0.043 -0.013  0.107 -0.017]


100,000 samples  [-0.035 -0.043 -0.013  0.108 -0.017]
8 samples        [0. 0. 0. 0. 0.]


With 100,000 samples the estimate lands right on the true gradient. With 8, which is about what you can afford when every sample is a whole generated answer, we get exactly zero. None of the 8 tries happened to be right, so there was nothing to push up. No trick recovers a signal from tries that all failed, which is why RLVR needs prompts the model sometimes gets right. But even when some tries do pass, a handful of samples gives a noisy estimate, and comparing each try against its siblings is what tames that noise.

## Compare each answer with its group

![Eight rewards for one prompt, their mean, and the advantages. Answers above the mean get a positive advantage and are pushed up, answers below get a negative one and are pushed down.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/rlvr-advantage.svg)

Say the checker gives an answer 0.67. Whether that answer should go up depends on the prompt. On a prompt the model usually fails, 0.67 is good news, and on one it usually gets right, it's a step backwards. So GRPO judges each answer against a group of $G$ answers to the same prompt, as in the picture: eight rewards, their mean, and how far each one sits from the mean in units of the group's spread.

$$
A^{(i)} = \frac{R^{(i)} - \operatorname{mean}_j R^{(j)}}{\operatorname{std}_j R^{(j)} + 10^{-6}}.
$$

Beat your group and you're pushed up, fall short and you're pushed down, and dividing by the spread puts easy and hard prompts on one scale. A group whose answers all scored the same gets an advantage of zero everywhere, so the all-failed case still teaches nothing. (The $10^{-6}$ only keeps us from dividing by zero.)

Subtracting the mean doesn't change which way the gradient points on average (the comment in the next cell says why). What it should change is the noise. For this check we'll only subtract the group mean, without dividing by its spread. Across 4,000 groups of 8 answers, does that bring the estimate closer to the true gradient?

In [7]:
# Subtracting a baseline that doesn't depend on the sampled answer leaves the average
# gradient unchanged, since E[grad log pi(a)] = grad sum_a pi(a) = grad 1 = 0. The group
# mean includes the answer's own reward, which shrinks the gradient by (G-1)/G without
# turning it; dew.rl.rloo_advantage leaves the answer out.
exact = jax.grad(expected_reward)(start_theta)


def squared_error(key, baseline, groups=4000, size=8):
    """How far the 8-sample estimate lands from the exact gradient, on average."""
    answers = jax.random.categorical(key, start_theta, shape=(groups, size))
    weight = reward[answers]
    if baseline:
        weight = weight - weight.mean(axis=1, keepdims=True)
    push = jax.nn.one_hot(answers, K) - jax.nn.softmax(start_theta)
    estimates = jnp.mean(weight[..., None] * push, axis=1)
    return float(jnp.mean(jnp.sum((estimates - exact) ** 2, axis=1)))


key = jax.random.key(3)
print(f"squared error with R:         {squared_error(key, baseline=False):.4f}")
print(f"squared error with R - mean:  {squared_error(key, baseline=True):.4f}")

squared error with R:         0.0137
squared error with R - mean:  0.0102


It does. Across 4,000 groups of 8 answers, subtracting each group's mean lowers the average squared error from 0.0137 to 0.0102, about a quarter less.

## From one choice to a whole answer

![One completion as the loss reads it: the prompt 7 2 9 4 with mask 0, and the written digits 5 6 1 2 9 0 with mask 1, each carrying the completion's advantage.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/rlvr-sequence.svg)

A real answer is a sequence of choices, one per token, and its six digits have to share one reward. Because an answer's log-probability is the sum of its tokens' log-probabilities, we give every token the model wrote the same advantage, its answer's, as in the picture. The model didn't choose the prompt, and a mask $m$ (1 on written tokens, 0 everywhere else) leaves those tokens out. With $\ell_t = \log \pi_\theta(y_t \mid q, y_{<t})$ for one token, Dew minimizes the average over every written token in the batch:

$$
L(\theta) = -\frac{\sum_{i,t} m^{(i)}_t A^{(i)} \ell^{(i)}_t}{\sum_{i,t} m^{(i)}_t}.
$$

Below is one real group from the batch above, the 8 answers our untrained model wrote for one prompt, colored by the sign of their advantage.

In [8]:
# A packed batch has its own row order; session_index says which answer a row holds.
order = np.argsort(batch["session_index"][:, 0])
ids = batch["input_ids"][order]
written = batch["response_mask"][order] > 0
advantages = batch["advantages"][order][:, -1]
truths = [str(row[PROMPT_LENGTH - 1]) for row in ids]
completions = [" ".join(map(str, row[mask])) for row, mask in zip(ids, written)]
rewards = np.array([counts_up("count", text, truth, "")
                    for text, truth in zip(completions, truths)])


def sign(advantage):
    if advantage == 0:
        return "surface"
    return "accent_fill" if advantage > 0 else "warm_fill"


group = range(GROUPS)
figures.tokens([[str(token) for token in ids[i]] for i in group],
               fills=[["none"] * PROMPT_LENGTH + [sign(advantages[i])] * NEW_TOKENS
                      for i in group],
               notes=[f"R {rewards[i]:.2f}   A {advantages[i]:+.2f}" for i in group])

Figure(447.6 x 282, 8 shapes)

Most answers score 0, as you'd expect from a random table. The two with 2 of their 6 digits right get +1.50, and the zeros all get −0.68. The answer that scored 0.17 is hardly good, but it beat this group's mean, so it still gets a small push up (+0.41).

The next cell checks Dew against what we described: the advantages from the checker's scores in NumPy, and the gradient by hand, where a token read after digit $r$ adds $\mathrm{onehot}(y) - \mathrm{softmax}(W_r)$ to row $r$ of the table, times its advantage. At the starting weights the ratio and the KL penalty of the next two sections change nothing, so that's the whole gradient.

In [9]:
grouped = rewards.reshape(-1, GROUPS)
centred = grouped - grouped.mean(axis=1, keepdims=True)
ours = (centred / (grouped.std(axis=1, ddof=1, keepdims=True) + 1e-6)).reshape(-1)
np.testing.assert_allclose(advantages, ours, rtol=1e-5, atol=1e-6)
print("the advantages match")

# Besides the batch, the loss reads the step number, a random key and the EMA weights.
step = Step(step=jnp.zeros((), jnp.int32), key=jax.random.key(0), ema=start)


def loss(params, batch):
    statistics, _ = objective.loss(params, batch, step)
    return statistics.mean()[0]


dews = np.asarray(jax.grad(loss)(start, batch)["params"]["table"])
table = np.asarray(start["params"]["table"], np.float64)
mask, tokens = batch["response_mask"], batch["input_ids"]
by_hand = np.zeros_like(table)
for row, column in zip(*np.nonzero(mask)):
    read, wrote = tokens[row, column - 1], tokens[row, column]
    probs = np.exp(table[read] - np.logaddexp.reduce(table[read]))
    advantage = batch["advantages"][row, column]
    by_hand[read] -= advantage / mask.sum() * (np.eye(VOCAB)[wrote] - probs)
np.testing.assert_allclose(dews, by_hand, rtol=1e-4, atol=1e-7)
print(f"the gradients match, largest difference {np.abs(dews - by_hand).max():.1e}")

the advantages match


the gradients match, largest difference 2.4e-09


Both match, and the gradients agree to 2.4e-09, about what float32 rounding leaves.

## Taking more than one step per batch

![The clipped objective against the ratio. With a positive advantage it rises until the ratio reaches 1.2 and is flat after; with a negative one it is flat below 0.8 and past the dual clip at 3. Where it is flat, the clipped term gives the token no gradient.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/rlvr-clip.svg)

Generating answers is the slow part, so we'd like more than one gradient step per batch, and we take two (`UPDATES`). But after the first step the model has moved, and the batch came from the model as it was before. Can we still train on it?

PPO ([Schulman et al., 2017](https://arxiv.org/abs/1707.06347)) weights each token by the ratio $\rho_t = \pi_\theta(y_t) / \pi_\text{old}(y_t)$, how much more or less likely the current model finds it than the model that wrote it, and stops pushing once that ratio has moved far enough the way the advantage asks:

$$
\max\big(-\rho_t A,\; -\operatorname{clip}(\rho_t, 0.8, 1.2)\, A\big).
$$

In the picture, a positive advantage keeps improving the objective until the ratio reaches 1.2. Past that the line is flat, so while the ratio stays above 1.2 the token gets no gradient from the reward term. A negative advantage does the same below 0.8. For negative advantages Dew also caps the term at $-3A$, the dual clip of [Ye et al., 2020](https://arxiv.org/abs/1912.09729), so one token with a runaway ratio can't take over a step. The next cell checks the clipped terms at a few ratios, then takes one large Adam step and counts the tokens the clip holds.

In [10]:
ratios = np.array([0.5, 0.9, 1.1, 1.5, 4.0])
for advantage in (1.0, -1.0):
    formula = np.maximum(-ratios * advantage, -np.clip(ratios, 0.8, 1.2) * advantage)
    if advantage < 0:
        formula = np.minimum(formula, -3.0 * advantage)
    terms, _ = clipped_surrogate_terms(jnp.log(ratios)[None], jnp.asarray([advantage]),
                                       jnp.ones((1, ratios.size)))
    np.testing.assert_allclose(terms[0], formula, rtol=1e-6)
    print(f"A = {advantage:+.0f}: terms {np.asarray(terms[0])} at ratios {ratios}")

adam = optax.adam(0.2)
update, _ = adam.update(jax.grad(loss)(start, batch), adam.init(start), start)
_, aux = objective.loss(optax.apply_updates(start, update), batch, step)
held = float(aux.metrics["actor/pg_clipfrac"])
print(f"after one step, the clip holds {held:.0%} of the written tokens")

A = +1: terms [-0.5 -0.9 -1.1 -1.2 -1.2] at ratios [0.5 0.9 1.1 1.5 4. ]
A = -1: terms [0.8 0.9 1.1 1.5 3. ] at ratios [0.5 0.9 1.1 1.5 4. ]


after one step, the clip holds 28% of the written tokens


The terms follow the formula. With a positive advantage they stop at −1.2 past a ratio of 1.2, and with a negative one they sit at 0.8 below 0.8 and stop at 3 when the dual clip takes over. After one large step, 28% of the written tokens are past the clip, so a second pass at these weights would give them no push from the reward.

## Keeping the model close to where it started

![The k3 estimate of the KL divergence against the policy's probability over the reference's. It is zero where they agree and rises on both sides, so its gradient pulls a drifted token back.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/rlvr-kl.svg)

GRPO's loss has one more term. While it chases the reward, the model can drift far from where it started, so GRPO adds $\beta$ times a per-token estimate of the KL divergence from the starting policy $\pi_\text{ref}$:

$$
k_3 = \frac{\pi_\text{ref}}{\pi_\theta} - \log \frac{\pi_\text{ref}}{\pi_\theta} - 1 \;\ge 0.
$$

It's zero when the two policies agree and grows as the model drifts either way, so its gradient points back towards the start, like the ball in the picture. Where the clip has flattened the reward term, this one can still pull, unless it has reached its own cap. Dew caps it at 10, as verl does, so one token the model has nearly ruled out can't swamp the loss; the last pair in the next cell is past that cap.

In [11]:
policy_probs = np.array([0.1, 0.3, 0.5, 0.9, 0.01])
reference_probs = np.array([0.3, 0.3, 0.25, 0.5, 0.2])
d = np.clip(np.log(reference_probs / policy_probs), -20, 20)
# Dew caps k3 at 10, and the log-ratio inside it at ±20, as verl does.
estimate = k3_kl(jnp.log(policy_probs), jnp.log(reference_probs))
capped = np.minimum(np.exp(d) - d - 1, 10)
np.testing.assert_allclose(estimate, capped, rtol=1e-5, atol=1e-7)
print("formula:", np.exp(d) - d - 1)
print("k3_kl:  ", np.asarray(estimate))

formula: [ 0.901  0.     0.193  0.143 16.004]
k3_kl:   [ 0.901  0.     0.193  0.143 10.   ]


The first four values match the formula. For the last pair the formula gives 16.0, and Dew stops at the cap of 10.

## Training it

Now we put the pieces together. Each step draws fresh prompts, samples and scores `GROUPS` answers to each, and takes `UPDATES` Adam steps on the batch. We record each batch's mean reward, and every few steps the model's table, to watch where the probability goes.

In [12]:
params, optimizer = start, optax.adam(LEARNING_RATE)
state = optimizer.init(params)
gradient = jax.jit(jax.grad(loss))
history, tables = [], []
every = max(STEPS // 10, 1)
for index in range(STEPS):
    if index % every == 0:
        tables.append((index, jax.nn.softmax(params["params"]["table"], axis=-1)))
    key = jax.random.fold_in(jax.random.key(2), index)
    now = SimpleNamespace(variables=params, updates=index)
    batch = rollout(now, prompt_batch(rng), key)
    history.append(rollout.metrics["reward/mean"])
    for _ in range(UPDATES):
        update, state = optimizer.update(gradient(params, batch), state, params)
        params = optax.apply_updates(params, update)
tables.append((STEPS, jax.nn.softmax(params["params"]["table"], axis=-1)))

figures.curve({"mean reward": history}, x_label="rollout", y_label="mean reward",
              y_range=(0, 1), smooth=max(STEPS // 15, 1))

Figure(460 x 290, 20 shapes)

The mean reward starts near 0.1, about what random digits get you, and ends at 0.99. The table shows what the model learned. Each row is the digit it just read, each column the digit it writes next, and the outlined cells are the right answers; the probability collects in them as training goes on.

In [13]:
digits = [str(d) for d in range(VOCAB)]
figures.heatmap([np.asarray(probs) for _, probs in tables], rows=digits, columns=digits,
                captions=[f"rollout {index}" for index, _ in tables],
                x_label="next digit", y_label="digit read",
                marks=[(d, (d + 1) % VOCAB) for d in range(VOCAB)])

Figure(348 x 384, 242 shapes)

In [14]:
prompt = jnp.asarray([[7, 2, 9, 4]], jnp.int32)
# Temperature 0 takes the most likely digit at every step.
greedy = generate(policy, params, prompt, NEW_TOKENS, key=0,
                  sampling=Sampling(temperature=0))
print("7 2 9 4 ->", greedy.tokens[0, PROMPT_LENGTH:])

7 2 9 4 -> [5 6 7 8 9 0]


Ask the trained model for its single most likely answer to `7 2 9 4` and it counts up, `5 6 7 8 9 0`, the answer we started the notebook with.

## What changes with a real model

The loss stays the same. A transformer gets its logits from a much deeper network, but `GRPOObjective` computes the same per-token terms we just checked. Everything around the loss gets bigger. The checker runs real programs, so it needs a sandbox (`dew.rl.sandbox`). While the trainer updates the model, a separate engine (Dew's `Server`, vLLM or SGLang) generates answers, so they can be one update old by the time we train on them. The example scores them again with the weights the update starts from, and caps each token's importance weight with `behavior_importance`. And long or multi-turn episodes are packed into rows with masks, which [tutorial 11](11-llms-in-an-agent-harness.ipynb) walks through.

[`examples/train_rlvr.py`](https://github.com/AshishKumar4/dew/blob/main/examples/train_rlvr.py) puts all of this together and trains a small instruct model to write Python programs that pass their tests. [Post-training](../docs/concepts/post_training.md) covers Dew's other objectives.